# EADS Killer Demo

Run the **Enterprise AI Decision Systems** companion end-to-end in Colab.

What you will see:
1. A single supply-chain decision from signals to verdict.
2. A reproducible benchmark across two scenarios.
3. An auditable `results.json` file.

In [ ]:
!pip install git+https://github.com/eads-research/enterprise-ai-decision-systems.git

## 1. One decision in 5 lines

In [ ]:
from eads.core.pipeline import DecisionPipeline
from eads.core.types import DecisionRequest
from eads.decision.decision import DecisionEngine
from eads.governance import GovernanceLayer
from eads.synthetic_data import SupplyChainGenerator

request = DecisionRequest(
    request_id='demo',
    goal='decide replenishment order for SKU-1001',
    signals=SupplyChainGenerator(seed=42).generate(3),
)
record = DecisionPipeline(governance=GovernanceLayer(), decision_engine=DecisionEngine()).run(request)

print('Approved:', record.verdict.approved)
print('Reason:', record.verdict.reason)
print('Actions:', record.decision.actions)
print('Trace steps:', [t['step'] for t in record.trace])

## 2. Full supply-chain benchmark

In [ ]:
from eads.core.pipeline import DecisionPipeline
from eads.core.types import DecisionRequest
from eads.decision.decision import DecisionEngine
from eads.evaluation import Benchmark
from eads.governance import GovernanceLayer
from eads.synthetic_data import SupplyChainGenerator

gen = SupplyChainGenerator(seed=42)
engine = DecisionEngine()
governance = GovernanceLayer()
pipeline = DecisionPipeline(governance=governance, decision_engine=engine)

scenarios = [
    {
        'id': 'sc-1',
        'request': DecisionRequest(
            request_id='sc-1',
            goal='replenish SKU-1001',
            signals=gen.generate(3),
            policy_snapshot={'max_order_quantity': 1000, 'unit_price': 10.0},
        ),
    },
    {
        'id': 'sc-2',
        'request': DecisionRequest(
            request_id='sc-2',
            goal='emergency purchase for SKU-1001',
            signals=gen.generate(3),
            policy_snapshot={'max_order_quantity': 50, 'unit_price': 10.0},
        ),
    },
]
benchmark = Benchmark(
    pipeline,
    scenarios,
    output_dir='benchmarks/results',
    metadata={'example': 'killer_demo', 'version': '1.0.0'},
)
print(benchmark.run())

## Next steps

- Read the demo walkthrough: [`docs/wow_demo.md`](../docs/wow_demo.md)
- Run more examples: `examples/healthcare.py`, `examples/finance.py`, `examples/it_operations.py`, `examples/customer_support.py`
- See the full docs: `https://eads-research.github.io/enterprise-ai-decision-systems`